**Important: read the following before you hit "Run all"**

Model checkpoints will be saved and loaded locally.

This notebook uses Google Colab. So, local means local disk used in the current Google Colab session.

To persist model checkpoints across sessions, this notebook saves the local copy to Google Drive after training.

To load a model back, this notebook copies model checkpoints from Google Drive back to Google Colab.

To transfer data between Google Colab and Google Drive, Google Drive is mounted under "/content/drive" and our data can be accessed via "/content/drive/MyDrive" in Google Colab.

**Checkpoints needed for training:**

R1 checkpoints on Google Drive: /checkpoints/r1_checkpoints

E1 checkpoints on Google Drive: /checkpoints/sft_sst2_gpt2_E1-1_2logits

**Dataset after creation:**

Preference dataset on Google Drive: /datasets/r2_preference_dataset

**Note**:

1) Change these locations and files operations to match your environments as necessary. For example, you may mount your Google Drive, then load and save checkpoints directly from Google Drive. Though, load & save on local + copy before load & save after training between Google Colab and Google Drive as described above is preferred.

In [ ]:
# First, mount Drive (run in a cell)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copy R1 & E1 from Google Drive to local

!cp -r drive/MyDrive/checkpoints/r1_checkpoints/ .

!cp -r drive/MyDrive/checkpoints/sft_sst2_gpt2_E1-1_2logits .


In [ ]:
import json
import os
import random
from pathlib import Path

import torch
import torch.nn.functional as F

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2Model,
    set_seed,
)
from torch import nn


In [ ]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------

MODEL_NAME = "openai-community/gpt2"

R1_CHECKPOINT = "./r1_checkpoints/r1-sft-best"

# Change this to the actual E1 checkpoint location.
E1_CHECKPOINT = "./sft_sst2_gpt2_E1-1_2logits"
E1_CHECKPOINT_WEIGHTS_FILE = "final_model_weights.pth"

OUTPUT_DIR = Path("./r2_preference_dataset")

SEED = 42

MAX_TRAIN_PROMPTS = 2000
MAX_VALIDATION_PROMPTS = 400

NUM_CANDIDATES = 4

MAX_NEW_TOKENS = 40
MAX_LENGTH = 128

TEMPERATURE = 0.9
TOP_P = 0.95

MIN_REWARD_MARGIN = 0.10

PAD_TOKEN_ID = 50256
EOS_TOKEN_ID = 50256

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

set_seed(SEED)

print(f"Device: {DEVICE}")

Device: cuda


In [ ]:
# Reuse the R1 prompt transformation

def transform_example(example, rng):
    sentence = example["sentence"].strip()

    sentiment = {
        0: "negative",
        1: "positive"
    }[example["label"]]

    words = sentence.split()

    if len(words) < 8:
        return None

    split_ratio = rng.uniform(0.5, 0.7)
    split_index = int(len(words) * split_ratio)

    split_index = max(4, split_index)
    split_index = min(len(words) - 3, split_index)

    prefix = " ".join(words[:split_index])

    prompt = (
        f"Sentiment: {sentiment}\n"
        f"Continue the following review:\n"
        f"{prefix}"
    )

    return {
        "prompt": prompt,
        "prefix": prefix,
        "sentiment": sentiment,
    }


def generate_prompt_examples(dataset_split, max_examples, seed=42):
    rng = random.Random(seed)

    for example in dataset_split:
        transformed = transform_example(example, rng)

        if transformed is not None:
            yield transformed

        # Limit the number of valid transformed prompts.
        # The counter is maintained by the caller below.

def collect_prompts(dataset_split, max_examples, seed=42):
    prompts = generate_prompt_examples(
        dataset_split,
        max_examples,
        seed
    )

    results = []

    for example in prompts:
        results.append(example)

        if len(results) >= max_examples:
            break

    return results

In [ ]:
# Load R1 - the base gen model to tune using DPO and to generate continuation candidates (e.g. 4) for preference pairs of a prompt for DPO

r1_tokenizer = AutoTokenizer.from_pretrained(R1_CHECKPOINT)

if r1_tokenizer.pad_token is None:
    r1_tokenizer.pad_token = r1_tokenizer.eos_token

r1_model = GPT2LMHeadModel.from_pretrained(R1_CHECKPOINT)
r1_model.to(DEVICE)
r1_model.eval()

r1_model.config.pad_token_id = r1_tokenizer.pad_token_id
r1_model.config.use_cache = True

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
# Load E1 - the sentiment classifier, used to classify sentiment of texts to determine "chosen" and "rejected"
# can be thinked of as a reward model

# Load the tokenizer (it knows how to reconstruct itself from the folder)
reward_tokenizer = AutoTokenizer.from_pretrained(E1_CHECKPOINT)


# ============================================================
# Model
# ============================================================

class GPT2SentimentClassifier(nn.Module):

    def __init__(self, model_name, num_labels=2):
        super().__init__()

        # GPT-2 WITHOUT the language-model head.
        self.gpt2 = GPT2Model.from_pretrained(model_name)

        hidden_size = self.gpt2.config.hidden_size

        # Binary classification head:
        #
        # [B, 768] -> [B, 2]
        self.classifier = nn.Linear(
            hidden_size,
            num_labels,
        )


    def forward(self, input_ids, attention_mask, labels=None):

        # ----------------------------------------------------
        # GPT-2
        #
        # last_hidden_state:
        # [B, T, 768]
        # ----------------------------------------------------

        outputs = self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        hidden_states = outputs.last_hidden_state

        # ----------------------------------------------------
        # Find the position of the last real token
        #
        # attention_mask:
        #
        #   1 1 1 1 1 0 0
        #
        # last real token = position 4
        # ----------------------------------------------------

        sequence_lengths = attention_mask.sum(dim=1) - 1

        batch_indices = torch.arange(
            hidden_states.size(0),
            device=hidden_states.device,
        )

        # [B, 768]
        pooled_hidden_states = hidden_states[
            batch_indices,
            sequence_lengths,
        ]

        # ----------------------------------------------------
        # Classification head
        #
        # [B, 768] -> [B, 2]
        # ----------------------------------------------------

        logits = self.classifier(
            pooled_hidden_states
        )


        # ----------------------------------------------------
        # Classification loss
        # ----------------------------------------------------

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {
            "loss": loss,
            "logits": logits,
        }

# 2. Initialize our model class (we know the architecture)
reward_model = GPT2SentimentClassifier(
    MODEL_NAME,
    num_labels=2,
)

# 3. Load the saved weights
reward_checkpoint = torch.load(os.path.join(E1_CHECKPOINT, E1_CHECKPOINT_WEIGHTS_FILE), weights_only=True)
reward_model.load_state_dict(reward_checkpoint['model_state_dict'])
reward_model.to(DEVICE)
reward_model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2SentimentClassifier(
  (gpt2): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Linear(in_features=768, out_features=2, bias=True)
)

In [ ]:
# Generate candidate continuations from R1
# For each prompt, generate four different continuations using sampling.

@torch.inference_mode()
def generate_candidates(prompt):
    inputs = r1_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    input_length = inputs["input_ids"].shape[1]

    outputs = r1_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        num_return_sequences=NUM_CANDIDATES,
        pad_token_id=PAD_TOKEN_ID,
        eos_token_id=EOS_TOKEN_ID,
        repetition_penalty=1.1,
    )

    candidates = []

    for output in outputs:
        new_tokens = output[input_length:]

        continuation = r1_tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        )

        if len(continuation.strip().split()) < 3:
            continue

        candidates.append(continuation)

    # Remove duplicate continuations.
    candidates = list(dict.fromkeys(candidates))

    return candidates

In [ ]:
# Score candidates with E1
# The classifier must evaluate the complete review, not just the newly generated text.

# Fixed version, which returns each candidate's logits, probabilities, predicted class,
# and target-class reward, to provide suffice information for create_preference_pair

@torch.inference_mode()
def score_candidates(prefix, candidates, requested_sentiment):
    full_texts = [
        prefix + continuation
        for continuation in candidates
    ]

    inputs = reward_tokenizer(
        full_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    outputs = reward_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

    logits = outputs["logits"]

    probabilities = F.softmax(logits, dim=-1)

    target_label = {
        "negative": 0,
        "positive": 1,
    }[requested_sentiment]

    predictions = logits.argmax(dim=-1)

    results = []

    for i, candidate in enumerate(candidates):
        results.append({
            "continuation": candidate,
            "logits": logits[i].cpu().tolist(),
            "probabilities": probabilities[i].cpu().tolist(),
            "predicted_label": predictions[i].item(),
            "target_reward": probabilities[i, target_label].item(),
        })

    return results


"""
Less ideal implementation, because it does not require a chosen continuation to be predicted as the requested sentiment by E1
"""

@torch.inference_mode()
def score_candidates_1(prefix, candidates, requested_sentiment):
    full_texts = [
        prefix + continuation
        for continuation in candidates
    ]

    inputs = reward_tokenizer(
        full_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    outputs = reward_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

    # Extract logits according to the model's return format.
    if isinstance(outputs, dict):
        logits = outputs["logits"]
    elif hasattr(outputs, "logits"):
        logits = outputs.logits
    else:
        logits = outputs

    probabilities = F.softmax(logits, dim=-1)

    target_label = {
        "negative": 0,
        "positive": 1,
    }[requested_sentiment]

    rewards = probabilities[:, target_label]

    return rewards.cpu().tolist()

"""
Wrong implementation based on mistaking E1 model output:

logits = reward_model(input_ids=inputs["input_ids"],
  attention_mask=inputs["attention_mask"],)

as returning a tensor with shape [B, 2].
But our E1 classifier apparently returns something like:
{    "logits": tensor(...)}

Actual type & shape in case of 4 candidates of continuation generated
#   - Output type: <class 'dict'>
#   - Output keys: dict_keys(['loss', 'logits'])
#   - Logits shape: torch.Size([4, 2]) # 4 candidates

Consequently, F.softmax(logits, dim=-1) fails because logits is actually a dictionary and throws the following error:

AttributeError                            Traceback (most recent call last)

/tmp/ipykernel_1030/2704162554.py in <cell line: 0>()

     37

     38

---> 39 train_rows = build_preference_dataset(

     40     train_prompts,

     41     "Train"

4 frames

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py in softmax(input, dim, _stacklevel, dtype)

   2157         dim = _get_softmax_dim("softmax", input.dim(), _stacklevel)

   2158     if dtype is None:

-> 2159         ret = input.softmax(dim)

   2160     else:

   2161         ret = input.softmax(dim, dtype=dtype)

AttributeError: 'dict' object has no attribute 'softmax'

"""
@torch.inference_mode()
def score_candidates_0(prefix, candidates, requested_sentiment):
    full_texts = [
        prefix + continuation
        for continuation in candidates
    ]

    inputs = reward_tokenizer(
        full_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(DEVICE)

    logits = reward_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

    probabilities = F.softmax(logits, dim=-1)

    target_label = {
        "negative": 0,
        "positive": 1,
    }[requested_sentiment]

    rewards = probabilities[:, target_label]

    return rewards.cpu().tolist()

In [ ]:
# Verify  E1 model's output before rerunning the entire dataset generation

# Build train dataset for a quick test of E1 model's output bype
#   - Output type: <class 'dict'>
#   - Output keys: dict_keys(['loss', 'logits'])
#   - Logits shape: torch.Size([4, 2]) # 4 candidates

sst2 = load_dataset("stanfordnlp/sst2")

train_prompts = collect_prompts(
    sst2["train"],
    MAX_TRAIN_PROMPTS,
    seed=SEED
)

print(f"Train prompts: {len(train_prompts)}")


example = train_prompts[0]

print(f"example: {example}")

candidates = generate_candidates(example["prompt"])

inputs = reward_tokenizer(
    [
        example["prefix"] + candidate
        for candidate in candidates
    ],
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
).to(DEVICE)

with torch.inference_mode():
    outputs = reward_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

print("Output type:", type(outputs))

if isinstance(outputs, dict):
    print("Output keys:", outputs.keys())
    print("Logits shape:", outputs["logits"].shape)
elif hasattr(outputs, "logits"):
    print("Logits shape:", outputs.logits.shape)
else:
    print("Output shape:", outputs.shape)

"""
The expected logits shape is:
[B, 2]


For four candidates, it should be:
[4, 2]
"""

# Once this test passes, try the following
rewards = score_candidates(
    example["prefix"],
    candidates,
    example["sentiment"]
)

print("Candidates:", candidates)
print("Rewards:", rewards)

Train prompts: 2000
example: {'prompt': 'Sentiment: positive\nContinue the following review:\nthat loves its characters and communicates something', 'prefix': 'that loves its characters and communicates something', 'sentiment': 'positive'}
Output type: <class 'dict'>
Output keys: dict_keys(['loss', 'logits'])
Logits shape: torch.Size([2, 2])
Candidates: [' very personal about the human condition', ' even better than cinema has done for some time now']
Rewards: [{'continuation': ' very personal about the human condition', 'logits': [4.401423931121826, 5.362894058227539], 'probabilities': [0.2765839695930481, 0.7234160304069519], 'predicted_label': 1, 'target_reward': 0.7234160304069519}, {'continuation': ' even better than cinema has done for some time now', 'logits': [3.7046642303466797, 4.137606620788574], 'probabilities': [0.3934239149093628, 0.6065760850906372], 'predicted_label': 1, 'target_reward': 0.6065760850906372}]


In [ ]:
# Construct preference pairs

# Fixed version, to make sure a chosen candidate is the one that is predicted by E1 as the requested sentiment and
#  discard the prompt if there is all candidates are predicted incorrectly by E1
def create_preference_pair(example):
    candidates = generate_candidates(example["prompt"])

    if len(candidates) < 2:
        return None

    scored = score_candidates(
        example["prefix"],
        candidates,
        example["sentiment"],
    )

    target_label = {
        "negative": 0,
        "positive": 1,
    }[example["sentiment"]]

    opposite_label = 1 - target_label

    # Divide candidates according to E1's prediction.
    correctly_predicted = [
        item for item in scored
        if item["predicted_label"] == target_label
    ]

    incorrectly_predicted = [
        item for item in scored
        if item["predicted_label"] == opposite_label
    ]

    preference_type = (
      "correct_vs_incorrect"
      if incorrectly_predicted
      else "strong_vs_weak"
    )

    # We cannot create a valid chosen candidate if E1
    # considers every candidate to have the wrong sentiment.
    if not correctly_predicted:
        return None

    # Choose the candidate with the highest target-class probability.
    chosen = max(
        correctly_predicted,
        key=lambda item: item["target_reward"]
    )

    if incorrectly_predicted:
        # Prefer the strongest candidate from the wrong class.
        # This is a useful "hard negative" for DPO.
        rejected = min(
            incorrectly_predicted,
            key=lambda item: item["target_reward"]
        )
    else:
        # All candidates were classified correctly.
        # Select the weakest one as the rejected candidate.
        remaining = [
            item for item in correctly_predicted
            if item is not chosen
        ]

        if not remaining:
            return None

        rejected = min(
            remaining,
            key=lambda item: item["target_reward"]
        )

    reward_margin = (
        chosen["target_reward"] - rejected["target_reward"]
    )

    if reward_margin < MIN_REWARD_MARGIN:
        return None

    return {
        "prompt": example["prompt"],
        "chosen": chosen["continuation"],
        "rejected": rejected["continuation"],

        # Diagnostic fields.
        "sentiment": example["sentiment"],
        "chosen_reward": chosen["target_reward"],
        "rejected_reward": rejected["target_reward"],
        "reward_margin": reward_margin,

        "chosen_logits": chosen["logits"],
        "rejected_logits": rejected["logits"],

        "chosen_predicted_label": chosen["predicted_label"],
        "rejected_predicted_label": rejected["predicted_label"],

        "preference_type": preference_type,
    }

"""
Less ideal implementation along with score_candidates_1
"""
def create_preference_pair_1(example):
    candidates = generate_candidates(example["prompt"])

    if len(candidates) < 2:
        return None

    rewards = score_candidates(
        example["prefix"],
        candidates,
        example["sentiment"]
    )

    scored_candidates = sorted(
        zip(candidates, rewards),
        key=lambda item: item[1]
    )

    rejected, rejected_reward = scored_candidates[0]
    chosen, chosen_reward = scored_candidates[-1]

    reward_margin = chosen_reward - rejected_reward

    if reward_margin < MIN_REWARD_MARGIN:
        return None

    return {
        "prompt": example["prompt"],
        "chosen": chosen,
        "rejected": rejected,

        # Diagnostic fields; not required by DPOTrainer.
        "sentiment": example["sentiment"],
        "chosen_reward": chosen_reward,
        "rejected_reward": rejected_reward,
        "reward_margin": reward_margin,
    }

In [ ]:
# Build train and validation dataset

sst2 = load_dataset("stanfordnlp/sst2")

train_prompts = collect_prompts(
    sst2["train"],
    MAX_TRAIN_PROMPTS,
    seed=SEED
)

validation_prompts = collect_prompts(
    sst2["validation"],
    MAX_VALIDATION_PROMPTS,
    seed=SEED + 1
)

print(f"Train prompts: {len(train_prompts)}")
print(f"Validation prompts: {len(validation_prompts)}")


def build_preference_dataset(prompts, split_name):
    rows = []

    for i, example in enumerate(prompts):
        pair = create_preference_pair(example)

        if pair is not None:
            rows.append(pair)

        if (i + 1) % 50 == 0:
            print(
                f"{split_name}: processed {i + 1}/{len(prompts)}, "
                f"accepted {len(rows)}"
            )

    return rows


train_rows = build_preference_dataset(
    train_prompts,
    "Train"
)

validation_rows = build_preference_dataset(
    validation_prompts,
    "Validation"
)

print()
print(f"Accepted training pairs: {len(train_rows)}")
#print(f"Accepted validation pairs: {len(validation_rows)}")

Train prompts: 2000
Validation prompts: 400
Train: processed 50/2000, accepted 29
Train: processed 100/2000, accepted 54
Train: processed 150/2000, accepted 83
Train: processed 200/2000, accepted 116
Train: processed 250/2000, accepted 142
Train: processed 300/2000, accepted 172
Train: processed 350/2000, accepted 195
Train: processed 400/2000, accepted 227
Train: processed 450/2000, accepted 264
Train: processed 500/2000, accepted 292
Train: processed 550/2000, accepted 319
Train: processed 600/2000, accepted 350
Train: processed 650/2000, accepted 383
Train: processed 700/2000, accepted 415
Train: processed 750/2000, accepted 445
Train: processed 800/2000, accepted 477
Train: processed 850/2000, accepted 504
Train: processed 900/2000, accepted 536
Train: processed 950/2000, accepted 565
Train: processed 1000/2000, accepted 594
Train: processed 1050/2000, accepted 627
Train: processed 1100/2000, accepted 661
Train: processed 1150/2000, accepted 690
Train: processed 1200/2000, accepted

In [ ]:
# Save the datasetsI in two versions:
#   - 1. A DPO dataset containing exactly the three fields needed by TRL.
#   - 2. A diagnostic dataset containing reward scores and other information that will help us inspect preference quality.

# The resulting folder will look like:
""" r2_preference_dataset/
      train.jsonl
      validation.jsonl
      train_diagnostics.jsonl
      validation_diagnostics.jsonl
"""

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_split(rows, split_name):
    # Fields expected by TRL DPOTrainer.
    dpo_rows = [
        {
            "prompt": row["prompt"],
            "chosen": row["chosen"],
            "rejected": row["rejected"],
        }
        for row in rows
    ]

    dpo_dataset = Dataset.from_list(dpo_rows)

    dpo_dataset.to_json(
        str(OUTPUT_DIR / f"{split_name}.jsonl")
    )

    # Include reward diagnostics for inspection.
    diagnostic_dataset = Dataset.from_list(rows)

    diagnostic_dataset.to_json(
        str(OUTPUT_DIR / f"{split_name}_diagnostics.jsonl")
    )


save_split(train_rows, "train")
save_split(validation_rows, "validation")

print(f"Datasets saved under: {OUTPUT_DIR}")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Datasets saved under: r2_preference_dataset


In [ ]:
# Inspect the dataset before starting DPO training: both individual examples and aggregate statistics.
def inspect_dataset(rows, split_name):
    print()
    print("=" * 70)
    print(f"{split_name.upper()} DATASET INSPECTION")
    print("=" * 70)

    print(f"Number of preference pairs: {len(rows)}")

    if not rows:
        print("No preference pairs were generated.")
        return

    margins = [row["reward_margin"] for row in rows]

    chosen_rewards = [row["chosen_reward"] for row in rows]
    rejected_rewards = [row["rejected_reward"] for row in rows]

    print(
        f"Average chosen reward: "
        f"{sum(chosen_rewards) / len(chosen_rewards):.4f}"
    )

    print(
        f"Average rejected reward: "
        f"{sum(rejected_rewards) / len(rejected_rewards):.4f}"
    )

    print(
        f"Average reward margin: "
        f"{sum(margins) / len(margins):.4f}"
    )

    for sentiment in ["positive", "negative"]:
        subset = [
            row for row in rows
            if row["sentiment"] == sentiment
        ]

        print(f"{sentiment.capitalize()} pairs: {len(subset)}")

    print()
    print("SAMPLE PREFERENCE PAIRS")
    print("-" * 70)

    for i, row in enumerate(rows[:5]):
        print(f"\nExample {i + 1}")
        print(f"Requested sentiment: {row['sentiment']}")
        print(f"Prompt:\n{row['prompt']}")

        print(
            f"\nCHOSEN (reward={row['chosen_reward']:.4f}):"
            f"\n{row['chosen']}"
        )

        print(
            f"\nREJECTED (reward={row['rejected_reward']:.4f}):"
            f"\n{row['rejected']}"
        )

        print(f"\nReward margin: {row['reward_margin']:.4f}")
        print("-" * 70)


inspect_dataset(train_rows, "train")
inspect_dataset(validation_rows, "validation")


TRAIN DATASET INSPECTION
Number of preference pairs: 1178
Average chosen reward: 0.7485
Average rejected reward: 0.4798
Average reward margin: 0.2688
Positive pairs: 591
Negative pairs: 587

SAMPLE PREFERENCE PAIRS
----------------------------------------------------------------------

Example 1
Requested sentiment: positive
Prompt:
Sentiment: positive
Continue the following review:
that loves its characters and communicates something

CHOSEN (reward=0.7317):
 deeply personal --

REJECTED (reward=0.4428):
 deeply personal about them , even as a child

Reward margin: 0.2890
----------------------------------------------------------------------

Example 2
Requested sentiment: negative
Prompt:
Sentiment: negative
Continue the following review:
remains utterly satisfied to

CHOSEN (reward=0.7657):
 read it all as one big dumbed-down , overblown action sequence

REJECTED (reward=0.4236):
 have a one-hour show with three or four hours

Reward margin: 0.3421
---------------------------------

In [ ]:
from collections import Counter

print(Counter(
    row["preference_type"]
    for row in train_rows
))

print(Counter(
    row["preference_type"]
    for row in validation_rows
))

Counter({'correct_vs_incorrect': 638, 'strong_vs_weak': 540})
Counter({'correct_vs_incorrect': 119, 'strong_vs_weak': 96})


In [ ]:
# Save dataset to Google Drive

!cp -r ./r2_preference_dataset drive/MyDrive/datasets/